# Visualize Predictions on Map
This notebook loads saved model predictions (from `02_Machine_Learning_Modell.ipynb`) and visualizes predicted classes by neighborhood using the shapefile.

Required inputs:
- Predictions CSV: `../results/models/predictions_cph.csv` with columns: `cluster_id`, `prediction`.
- Shapefile: `../data/raw/neighborhood_shapefile/Nabolag_cph_fre_new.shp`.

In [1]:
# Imports
import os
import pandas as pd
import geopandas as gpd
import numpy as np
import leafmap.foliumap as leafmap
from IPython.display import HTML, display

In [ ]:
# Paths and load predictions + shapefile

# Prefer results/models; fallback to an interim test file if present
pred_primary = os.path.join('..', 'results', 'models', 'predictions_cph.csv')
pred_fallback = os.path.join('..', 'data', 'interim', 'predictions_test.csv')
shp_path = os.path.join('..', 'data', 'raw', 'neighborhood_shapefile', 'Nabolag_cph_fre_new.shp')

pred_path = pred_primary if os.path.exists(pred_primary) else pred_fallback
if not os.path.exists(pred_path):
    raise FileNotFoundError(f"Predictions file not found. Tried: {pred_primary} and {pred_fallback}.")

pred = pd.read_csv(pred_path)
print(f"Loaded predictions: {len(pred)} rows, columns: {list(pred.columns)}")

# Validate required columns
required_cols = {'cluster_id','prediction'}
missing = required_cols - set(pred.columns)
if missing:
    raise ValueError(f"Predictions missing required columns: {missing}")

# Load shapefile
gdf = gpd.read_file(shp_path)
print(f"Loaded shapefile: {len(gdf)} polygons, CRS: {gdf.crs}")

Loaded predictions: 285 rows, columns: ['cluster_id', 'y_true', 'y_pred']
Loaded shapefile: 1421 polygons, CRS: EPSG:25832
Loaded shapefile: 1421 polygons, CRS: EPSG:25832


In [ ]:
# Prepare merge and diagnostics
# Detect a suitable join key in the shapefile
join_key = None
for cand in ['munic_clus', 'cluster_id', 'Cluster_id']:
    if cand in gdf.columns:
        join_key = cand
        break
if join_key is None:
    raise ValueError("No suitable join key found in shapefile. Expect one of: munic_clus, cluster_id, Cluster_id.")

# Normalize join keys to string
gdf['cluster_id'] = gdf[join_key].astype(str)
pred['cluster_id'] = pred['cluster_id'].astype(str)

# Merge predictions onto polygons
merged = gdf.merge(pred[['cluster_id','prediction']], on='cluster_id', how='left')
print("Merged polygons:", len(merged))
print("Prediction value counts:\n", merged['prediction'].fillna('NoPrediction').value_counts(dropna=False))
print("Merged CRS:", merged.crs)

# Reproject to EPSG:4326 for web maps
try:
    if merged.crs is None or merged.crs.to_epsg() != 4326:
        merged = merged.to_crs(epsg=4326)
except Exception as e:
    print("CRS reprojection warning:", e)

Merged polygons: 1421
Status value counts:
 status
NoPrediction    1136
Correct          231
Incorrect         54
Name: count, dtype: int64
Merged CRS: EPSG:25832


In [ ]:
# Leafmap visualization
import leafmap.foliumap as leafmap

m = leafmap.Map(center=[55.6761, 12.5683], zoom=10)
m.add_basemap("CartoDB.Positron")

# Color mapping for predicted classes
style_dict = {
    0: {'color': '#2ca02c', 'fillColor': '#2ca02c', 'fillOpacity': 0.6},  # Class 0
    1: {'color': '#1f77b4', 'fillColor': '#1f77b4', 'fillOpacity': 0.6},  # Class 1
    2: {'color': '#ff7f0e', 'fillColor': '#ff7f0e', 'fillOpacity': 0.6},  # Class 2
    'NoPrediction': {'color': '#7f7f7f', 'fillColor': '#7f7f7f', 'fillOpacity': 0.3},
}

# Ensure a stable prediction column and fill missing
merged['prediction'] = merged['prediction']
geojson = merged.to_json()

# Tooltip fields (shown on hover)
tooltip_fields = [c for c in ['cluster_id', 'prediction'] if c in merged.columns]

def style_function(feature):
    val = feature['properties'].get('prediction', None)
    try:
        key = int(val) if val is not None and val == val else 'NoPrediction'
    except Exception:
        key = 'NoPrediction'
    return style_dict.get(key, style_dict['NoPrediction'])

# Add GeoJSON layer with tooltip
try:
    m.add_geojson(
        geojson,
        style_function=style_function,
        layer_name='Predicted Class',
        tooltip=tooltip_fields
    )
except Exception as e:
    print(f"Tooltip add warning: {e}. Adding layer without tooltip.")
    m.add_geojson(geojson, style_function=style_function, layer_name='Predicted Class')

# Add legend for classes
try:
    m.add_legend(
        title='Predicted Class',
        labels=['Class 0', 'Class 1', 'Class 2', 'NoPrediction'],
        colors=['#2ca02c', '#1f77b4', '#ff7f0e', '#7f7f7f']
    )
except Exception as e:
    print(f"Legend add warning: {e}.")

m.add_layer_control()

# Save to HTML
out_dir = os.path.join('..','results','figures')
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, 'prediction_map.html')
try:
    html = m.to_html()
    html = html.replace('.foliumtooltip {\n                            \n                        }', '.foliumtooltip { padding: 0; }')
    html = html.replace('.foliumtooltip{\n}', '.foliumtooltip { padding: 0; }')
    with open(out_path, 'w', encoding='utf-8') as f:
        f.write(html)
    print(f"Map saved to: {out_path}. Open this file in your web browser.")
except Exception as e:
    print(f"Save warning: {e}. Attempting direct save via leafmap.")
    try:
        m.to_html(out_path)
        print(f"Map saved to: {out_path} via direct save.")
    except Exception as e2:
        print(f"Direct save failed: {e2}.")

Tooltip add warning: folium.features.GeoJson() got multiple values for keyword argument 'tooltip'. Adding layer without tooltip.
Map saved to: ..\results\figures\prediction_map.html. Open this file in your web browser.
Map saved to: ..\results\figures\prediction_map.html. Open this file in your web browser.
